TP4 - Parte II

Grupo 6: Acosta, Armelini y Freire.

En este Jupyter Notebook resolvemos la Parte II del TP4 de la materia. Clasificación y regularización utilizando la EPH del 1T de 2005 y 2025.

Pregunta 2.1:

"Para cada año,partan la base respondieron en una base de prueba y una de entrenamiento(X_train, y_train, X_test, y_test) utilizando el comando train_test_split. La base de entrenamiento debe comprender el 70% de los datos, y la semilla a utilizar (random state instance) debe ser 101.

Establezca a desocupado como su variable dependiente en la base de entrenamiento (vector y). El resto de las variables serán las variables independientes (matriz X). Recuerden agregar la columna de unos (1)".

Respuesta: en los bloques de código debajo.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from pathlib import Path

### Carga de base de datos de trabajo
# Se configura la ruta a bases de datos
repo_root = Path.cwd()               # asume que el kernel se lanzó desde la raíz del repo
# Disclaimer: si genera error, se recomienda setear manualmente la ruta del repo y luego proceder con lo de abajo.
data_dir = repo_root / "Datasets"    # subcarpeta 'Datasets' del TP4 del repo

base_predic = pd.read_excel(data_dir / "Base_predic.xlsx")


In [6]:
### FILTRO RESPONDIERON ###

# Vemos los valores únicos de 'ESTADO' para comprobar que la categoría 0 esté formateada correctamente
print(base_predic['ESTADO'].unique())
# Vemos también los tipos de datos de esos valores únicos (esto revela si un 0 es número o texto)
print(base_predic['ESTADO'].apply(type).value_counts())

# Notamos que no hay ningún 0 (o su versión en texto) en la columna 'ESTADO', lo que indica que todos respondieron en el aglomerado en cuestión (Bahía Blanca - Cerri). Por lo tanto, el filtro "respondieron" incluye todos los registros en este caso.
# Para corroborar esto, se chequea la base mergeada y las bases orginales filtradas para el aglomerado. Para ello, cargamos nuevamente las bases mencionadas.
ind_2005 = pd.read_stata(data_dir / "Individual_t105.dta")             # Base Individuos 2005
ind_2005 = ind_2005[ind_2005["aglomerado"] == "Bahía Blanca - Cerri"]  # Filtrado por aglomerado
ind_2005.columns = ind_2005.columns.str.upper()                        # Nombres a mayúsculas
ind_2025 = pd.read_excel(data_dir / "usu_individual_T125.xlsx")        # Base Individuos 2025
ind_2025 = ind_2025[ind_2025["AGLOMERADO"] == 3]                       # Filtrado por aglomerado
base_merged = pd.read_excel(data_dir / "Base_merged.xlsx")             # Base mergeada
print("\n" + "="*20 + "\n")
# Verificamos si ESTADO sigue ahí en la base mergeada
if 'ESTADO' in base_merged.columns:
    print("\nLa columna ESTADO existe.")
    print("Conteo de nulos en ESTADO:", base_merged['ESTADO'].isna().sum())
    # Verificar si el 0 sigue existiendo
    print("Cantidad de ESTADO=0:", (base_merged['ESTADO'] == 0).sum())
else:
    print("\n La columna ESTADO no está.")
    print([col for col in base_merged.columns if 'ESTADO' in col])
# Auditoría de la variable ESTADO en las bases originales ---
print("=== DIAGNÓSTICO BASE INDIVIDUOS 2005 ===")
# 1. Ver qué valores existen y cuántos hay de cada uno
print(ind_2005['ESTADO'].value_counts(dropna=False).sort_index())
# 2. Ver la lista cruda de únicos para detectar textos escondidos (ej: '0' vs 0)
print("Valores únicos (lista):", list(ind_2005['ESTADO'].unique()))
print("\n" + "="*20 + "\n")
print("=== DIAGNÓSTICO BASE INDIVIDUOS 2025 ===")
# 1. Ver qué valores existen y cuántos hay de cada uno
print(ind_2025['ESTADO'].value_counts(dropna=False).sort_index())
# 2. Ver la lista cruda de únicos
print("Valores únicos (lista):", list(ind_2025['ESTADO'].unique()))

[1 3 4 2]
ESTADO
<class 'int'>    1833
Name: count, dtype: int64



La columna ESTADO existe.
Conteo de nulos en ESTADO: 0
Cantidad de ESTADO=0: 0
=== DIAGNÓSTICO BASE INDIVIDUOS 2005 ===
ESTADO
Entrevista individual no realizada (no respuesta al cuestion      0
Ocupado                                                         435
Desocupado                                                       52
Inactivo                                                        457
Menor de 10 años                                                122
Name: count, dtype: int64
Valores únicos (lista): ['Ocupado', 'Inactivo', 'Menor de 10 años', 'Desocupado']


=== DIAGNÓSTICO BASE INDIVIDUOS 2025 ===
ESTADO
1    339
2     16
3    327
4     85
Name: count, dtype: int64
Valores únicos (lista): [np.int64(1), np.int64(3), np.int64(4), np.int64(2)]


In [8]:
# Habiendo chequeado que entonces las bases utilizadas no tienen registros con ESTADO=0 para el aglomerado con el que estamos trabajando, se procede con el resto del análisis con la base de predicción limpia y con variables generadas que resulta de la Parte 1 del TP (Base_predic).

# Para verificación, filtramos y renombramos como respondieron y norespondieron
# Criterio: ESTADO == 0 son los que NO respondieron la condición de actividad
norespondieron = base_predic[base_predic['ESTADO'] == 0].copy()
respondieron = base_predic[base_predic['ESTADO'] != 0].copy()
print(f"=== Reporte de Filtrado ===")
print(f"Total observaciones iniciales: {len(base_predic)}")
print(f"Base 'norespondieron' (ESTADO=0): {len(norespondieron)}")
print(f"Base 'respondieron' (ESTADO!=0): {len(respondieron)} -> Se usará esta para modelar.")


### Limpieza previa al Split Train/Test

# Dejamos de lado los registros con ESTADO = 4 que indica Menor de 10 años, ya que no forman parte de la PEA y por ende no son relevantes para el ejercicio de predicción de desocupación.
respondieron = respondieron[respondieron['ESTADO'] != 4].copy()


### Split Train/Test para Machine Learning

base_ml = {}

for anio in [2005, 2025]:
    print(f"\n--- Procesando Año {anio} ---")
    
    # a) Filtramos por año sobre la base LIMPIA ('respondieron')
    df_anio = respondieron[respondieron['ANO4'] == anio].copy()
    
    # b) Definimos la Variable Dependiente (y)
    # 1 si es Desocupado (ESTADO==2), 0 en cualquier otro caso (Ocupado o Inactivo)
    df_anio['desocupado'] = (df_anio['ESTADO'] == 2).astype(int)
    y = df_anio['desocupado']
    
    # c) Definimos la Matriz X
    # Se debe sacar 'ESTADO' (porque es la dependiente) y variables administrativas.
    # Agrega aquí cualquier otra variable que NO deba ser predictora (ej. ingresos si quieres predecir sin ellos).
    vars_a_sacar = [
        'desocupado', 'ESTADO',            # La respuesta y su fuente directa (¡Data Leakage!)
        'CODUSU', 'NRO_HOGAR',             # Identificadores
        'COMPONENTE', 'H15',               # Identificadores intra-hogar
        'PONDERA', 'PONDII', 'PONDIIO',    # Ponderadores (no se usan como predictores)
        'PONDERA_IND', 'PONDERA_HOG', 
        'ANO4'                             # El año ya está fijo en el bucle
    ]
    # Quitamos solo las columnas que existan para evitar errores
    cols_drop = [c for c in vars_a_sacar if c in df_anio.columns]
    X = df_anio.drop(columns=cols_drop)
    
    # d) Agregamos la columna de Unos (Intercepto)
    X['intercepto'] = 1
    
    # e) Partición 70/30 con semilla 101
    X_train, X_test, y_train, y_test = train_test_split(   # Comando train_test_split
        X, y, 
        test_size=0.3,     # Proporción de partición para test
        random_state=101   # Semilla (random state instance)
    )
    
    # Guardamos todo en el diccionario
    base_ml[anio] = {
        'X_train': X_train,
        'y_train': y_train,
        'X_test': X_test,
        'y_test': y_test
    }
    
    print(f"Dimensiones X_train: {X_train.shape}")
    print(f"Dimensiones X_test:  {X_test.shape}")

=== Reporte de Filtrado ===
Total observaciones iniciales: 1833
Base 'norespondieron' (ESTADO=0): 0
Base 'respondieron' (ESTADO!=0): 1833 -> Se usará esta para modelar.

--- Procesando Año 2005 ---
Dimensiones X_train: (660, 338)
Dimensiones X_test:  (284, 338)

--- Procesando Año 2025 ---
Dimensiones X_train: (477, 338)
Dimensiones X_test:  (205, 338)


Pregunta 2.2:

"Expliquen como elegirían λ por validación cruzada. Detallen por qué no usarían el conjunto de prueba (test) para su elección".

Respuesta: en el documento de texto.

Pregunta 2.3:

"En validación cruzada, ¿cuáles son las implicancias de usar un k muy pequeño o uno muy grande? Cuando k = n (con n el número de muestras), ¿cuántas veces se estima el modelo?".

Respuesta: en el documento de texto.

Pregunta 2.4:

"Implementen la penalidad, L1 como la de LASSO y L2 como la de Ridge, para regresión logística usando la opción penalty y reporten la matriz de confusión, la curva ROC, los valores de AUC y de Accuracy para cada año.
    
[En la tutorial 6, vimos el método de regularización en regresión lineal donde la variable dependiente es numérica. En este caso, nuestra variable dependiente es binaria (ocupado, desocupado), por lo que usamos la regresión logística y aprovechamos la opción de penalidad para aplicar los métodos de regularización vistos en clase.].
    
¿Cómo cambiaron los resultados con respecto al TP3? ¿Mejor o empeoró la performance de regresión logística con regularización?".

Respuesta: en el código debajo y redacción en documento de texto.

Pregunta 2.5:

"Realicen un barrido en λ = 10^n con n ∈ {−5,−4,−3,...,+4,+5} y utilicen 10-fold
CV para elegir el λ óptimo en regresión logística con Ridge y con LASSO. ¿Qué λ
seleccionó en cada caso?

Usando la librería de seaborn, generen box plot mostran
dola distribución del error de predicción para cada λ. Cada box debe corresponder
a un valor de λ y contener como observaciones el error medio de validación para
cada partición.

Además, para la regularización LASSO, generen un line plot, pero
ahora graficando el promedio de la proporción de variables ignoradas por el mode
lo en función de λ, es decir la proporción de variables para las cuales el coeficiente
asociado es cero.

[Hint: a mayor penalidad, esperamos que más coeficientes sean 0, por lo tanto, esta figura debe tener una forma de “S”.]".

Respuesta: en el código debajo y redacción en documento de texto.

Pregunta 2.6:

"En el caso del valor óptimo de λ para LASSO encontrado en el inciso anterior, ¿qué
variables fueron descartadas? ¿Son las que hubieran esperado?

¿Tiene relación con lo que respondieron en el inciso 1 de la Parte I?".

Respuesta: en el código debajo y redacción en documento de texto.

Pregunta 2.7:

"Elijan alguno de los modelos de regresión logística donde hayan probado distintos parámetros de regularización y comenten: comparen los resultados de 2005 versus 2025, ¿qué método de regularización funcionó mejor: Ridge o LASSO?

Comenten mencionando el error cuadrático medio (ECM)".

Respuesta: en el código debajo y redacción en documento de texto.